<img src='./img/cng_thrivegeo_banner.png' class="center" width=95%>

# 🚀 Discover Dask

### Introduction

In *Discover Zarr*, you opened ERA5: 64 years of global weather, every 6 hours. You also counted the cost of different analyses: the temperature at Snowbird over a 64 year time period means reading **11,693 chunks**, about **11 GB**. That is more than your notebook can comfortably hold in memory. And a normal Python calculation uses only **one** of your processor cores, while the others wait.

[Dask](https://www.dask.org/) is a Python library for **parallel computing**. It splits bigger tasks into many small ones and runs them at the same time: on the cores of one machine, or on many machines in the cloud.

A simple way to picture Dask is to picture it as a 🍳 **kitchen**:

| In Dask | In the kitchen | What it means |
|---|---|---|
| **Chunks** | 🥕 Ingredients cut into portions | A large array is split into small blocks that fit in memory |
| **Task graph** | 📜 The recipe | A plan of every step and the order it happens in |
| **Scheduler** | 👩‍🍳 The head chef | Hands out the steps of the recipe |
| **Workers** | 🧑‍🍳 The cooks | Carry out the steps, many at the same time |
| **`.compute()`** | 🛎️ "Serve!" | Nothing is cooked until you ask for the result |



The last row is the important one. Dask is **lazy**: when you ask for a calculation, it only writes the recipe. The cooking happens when you ask for the result.

You have already met Dask without noticing. When you opened ERA5 in *Discover Zarr*, the summary showed a grey table with an **Array** and a **Chunk** column. That was a Dask array. In Zarr, a chunk was a piece of a store on disk. Here, a chunk is a piece of an array that one worker handles at a time.

In the following notebook, you start your own kitchen, write your first recipes with small arrays, and then scale up: from one year of ERA5 to decades of it. One question runs through it all: **how big should a chunk be?** You finish by asking ERA5 how November at Snowbird has changed over 64 years.

**Prerequisites:** basic Python and `numpy`, and *Discover Zarr*.

### By the end of this tutorial, you will be able to...

* ⚙️ **Start** a Dask cluster and follow its work on the dashboard
* 🧊 **Explain** how chunks become tasks, and why nothing is calculated until you ask for the result
* ▶️ **Choose** between `.compute()` and `.persist()`
* 📏 **Pick** a sensible working chunk size, not too small and not too big
* 📈 **Scale** a calculation from one year to decades of ERA5 data

### Notebook Outline

* [Start your Dask Cluster](#example1)
* [Example 1️⃣ : From numpy to Dask](#example2)
* [Exercise 1️⃣ : Sixteen chunks](#example3)
* [Example 2️⃣ : The lazy trap](#example4)
* [Example 4️⃣ : Scale up with ERA5](#example5)
* [Exercise 2️⃣ : 64 Novembers at Snowbird](#example6)
* [Takeaways](#example7)

<hr>

**Import required libraries**

In [ ]:
import time
import sys
import dask
import dask.array as da
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from dask.distributed import Client

Let us define a function that return the size of an array.

In [ ]:
# Define function to display variable size in MiB
def var_size(in_var):
    result = sys.getsizeof(in_var) / 1024/1024
    return (result)

<hr>

<a id='example1'></a>

## Start your Dask Cluster 🧑‍🍳

Before you can cook, you need a kitchen. In Dask, that is a **cluster**: a scheduler and a team of workers. You start one in three steps.

**1.** In the left sidebar, click the **Dask** tab:

<center><img alt="Alt text" src="./img/dask_dash.png" width=70%></center>

**2.** Click **+ NEW**. This creates your cluster:

<center><img alt="Alt text" src="./img/create_clus.png" width=30%></center>

**3.** Connect your notebook to the cluster. Scroll down in the sidebar, copy the **Scheduler Address**, and paste it into `Client(...)` in the cell below, in place of the example address. Or click the **< >** button next to **SCALE**, which inserts the whole cell for you:

<center><img alt="Alt text" src="./img/tcp_address.png" width=50%></center>

<div class="alert alert-block alert-info">

**Note**
You can also click on the <code>< ></code> button next to <b>SCALE</b>. That will automatically import the code cell below.
</div>

In [ ]:
client = Client("tcp://______")   # replace the address with your own
client

The box describes your kitchen. Read it line by line:

| In the box | Your value | In the kitchen |
|---|---|---|
| **Workers** | 4 | Four cooks |
| **Threads** per worker | 2 | Each cook has two hands, so **8 tasks** can run at the same time |
| **Memory** per worker | 7.8 GB | Each cook's worktop. A chunk must fit on it, with room to spare |
| **Total memory** | about 31 GB | All the worktops together |

*Note: your numbers might be different. Depending on the specifciations of your own machine.*

The **dashboard** is your window into the kitchen. Click a panel name in the Dask sidebar to open it, and drag it next to your notebook. Keep these three open while you work:

* 📊 **Task Stream**: one row per thread, one coloured bar per task. Gaps mean cooks waiting.
* ⏳ **Progress**: how many tasks of each kind are done.
* 🧠 **Workers Memory**: how full each worktop is. Orange or grey bars mean trouble.

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. How many tasks can your cluster run at exactly the same time?<br>
    ✏️ : **8**: 4 workers × 2 threads.
- ➡️ 2. In *Discover Zarr*, you saw that ERA5's 2 m temperature is about 11 GB. Could one worker hold all of it? Could the whole cluster?<br>
    ✏️ : One worker, no (7.8 GB). The whole cluster could just about hold it (about 31 GB), but that would leave little room for calculating. That is why Dask works **chunk by chunk**. It also gives the first rule for chunk size: a chunk must fit on one worktop, with room to spare.
</div>

<a id='example2'></a>

## Example 1️⃣: From numpy to Dask

In this example, you use an array so small that you can see **every single step** Dask plans: 4 × 4 numbers. You ask for the sum of all 16 numbers three times: with the array as one chunk, as four chunks, and as sixteen.

### ➡️ One chunk

Start with a normal `numpy` array:

In [ ]:
np_array = np.array([
        [ 12,  24,  36,  48],
        [ 60,  72,  84,  96],
        [ 108, 120, 132, 144],
        [156, 168, 180, 192]
        ])
np_array

Now turn it into a Dask array with `da.from_array()`:

In [ ]:
da_array = da.from_array(np_array)
da_array

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ How does the Dask array look different from the `numpy` array? Where are the numbers?<br>
    ✏️ : Dask doesn't print the numbers. It shows the same grey table you saw for ERA5 in *Discover Zarr*: the whole **Array** next to one **Chunk**, and a small picture of how the array is cut. Here there is only **1 chunk**: the whole array is one portion. A Dask array describes *how to get* the data, not the data itself.
</div>

Ask for the sum and look at what you get back:

In [ ]:
total = da_array.sum()
total

No number. `total` is still a Dask array: a **recipe** for the sum, not the sum itself. You can draw the recipe (TaskGraph) with `.visualize()` and count its tasks with `len(total.dask)`:

In [ ]:
print("Number of tasks:", len(total.dask))
total.visualize(rankdir="LR")

Read the graph from left to right. Each box is a **task** and each arrow means "needs the result of":

1. 📦 **array**: take the portion
2. ➕ **sum**: add up the numbers in that portion
3. 🍽️ **sum-aggregate**: put the final result together

Three tasks, one after the other. With a single portion, there's nothing to share out: one cook does all the work while the others watch.

### ➡️ Four chunks

Now cut the same array into four chunks of 2 × 2 with `chunks=(2, 2)`:

In [ ]:
da_quarters = da.from_array(np_array, chunks=(2, 2))
da_quarters

In [ ]:
total_quarters = da_quarters.sum()

print("Number of tasks:", len(total_quarters.dask))
total_quarters.visualize(rankdir="LR")

The task graph has grown from 3 to **9 tasks**, and it now has a different shape:

- 4 × 📦 take a chunk
- 4 × ➕ sum that chunk
- 1 × 🍽️ add the four partial sums together

Look at the four ➕ tasks: **none of them depends on another**. Four cooks can each sum one quarter at the same time, and the head chef adds up their four answers at the end. That is where Dask's speed comes from: **chunks create tasks that can run side by side**.

### ➡️ Serve!

So far, nothing has been calculated. Let's now serve the menue with the function `.compute()`, and compare with `numpy`:

In [ ]:
print("Dask: ", total_quarters.compute())
print("numpy:", np_array.sum())

Same answer, as it should be. `.compute()` runs the task graph on your workers and gives you back a normal Python or `numpy` value. In *Discover Zarr*, `.load()` did the same job for the ERA5 data.

For big calculations, the full task graph can have thousands of boxes. `.dask.visualize()` draws a **summary** instead, with one box per *kind* of step. You will need it later with ERA5:

In [ ]:
total_quarters.dask.visualize()

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. With four chunks, which tasks can run at the same time, and which must wait?<br>
    ✏️ : The four 📦 tasks and the four ➕ tasks can each run side by side. The 🍽️ task must wait until all four sums are ready.
- ➡️ 2. Your cluster can run 8 tasks at once. How many of them are busy during the ➕ step?<br>
    ✏️ : Only **4**. There are only four chunks, so half your kitchen is idle. More chunks would keep more cooks busy.
</div>

<hr>

<a id='example3'></a>

## Exercise 1️⃣ : Sixteen chunks


More chunks keep more cooks busy. So why not give every number its own chunk?

<div class="alert alert-block alert-warning">

**📝 Instructions**

**Predict first.** With `chunks=(1, 1)`, how many chunks does the array have? And roughly how many tasks does `.sum()` need? As a rough rule, every chunk needs one task to take it and one to sum it, plus a few more to combine the results.

**Then check.** Create the array, and print its grid of chunks (`.numblocks`), its number of chunks (`.npartitions`) and the number of tasks for `.sum()`.

**Hint:** `len(x.sum().dask)` counts the tasks, as in the cells above.
</div>

In [ ]:
da_sixteen = da.from_array(np_array, chunks=(1, 1))

print("Grid of chunks:  ", da_sixteen.numblocks)
print("Number of chunks:", da_sixteen.npartitions)
print("Number of tasks: ", len(da_sixteen.sum().dask))

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. How close was your prediction?<br>
    ✏️ : The array has **16 chunks** in a 4 × 4 grid, and the sum needs **37 tasks**: 16 to take a chunk, 16 to sum it, and 5 to combine the results.
- ➡️ 2. Is one number per chunk a good idea?<br>
    ✏️ : Not for a real dataset. 16 chunks keep all 8 threads busy, but the task graph has grown to 37 tasks just to add up 16 numbers. Each task costs a little time to plan and hand out, so the kitchen spends more time organising than cooking. **More chunks mean more parallel work, but also more tasks to manage.**
</div>

<div class="alert alert-block alert-info">

**Note**

📏 How big should a chunk be?

**First answer: not too small.** Every chunk adds tasks, and every task has to be planned and handed out. In the next section, you see how large Dask makes its chunks when you leave the choice to it.
</div>

<hr>

<a id='example4'></a>

## Example 2️⃣ :  The lazy trap

Now try something bigger: an array of 10,000 × 12,000 ones, almost **1 GB** of numbers. You race `numpy` against Dask, find out that the first result is a trap, and then make the array ten times bigger, more than one worker can hold.

Keep the dashboard's **Task Stream** and **Workers Memory** panels visible while you run this example.


### ➡️ A bigger array

In [ ]:
shape = (10000,12000)

ones_np = np.ones(shape)
ones_da = da.ones(shape)
ones_da

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. You didn't set `chunks=`. How did Dask cut the array anyway?<br>
    ✏️ : Dask chose for you: chunks of **4096 × 4096**, which gives a grid of 3 × 3 = **9 chunks**. The ones along the edges are smaller.
- ➡️ 2. How big is one chunk compared with the whole array?<br>
     ✏️ : About **128 MiB** per chunk, compared with about **916 MiB** for the whole array. By default, Dask aims for chunks of around 128 MiB: big enough to be worth a task, and small enough to fit easily on a worker.

</div>

### ➡️ The comparison

Calculate the mean of both arrays and time each one with `%%time`:

In [ ]:
%%time
ones_np.mean()

In [ ]:
%%time
mean_da = ones_da.mean()
mean_da

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ What do you observe? What computation is faster and why?<br>
    ✏️ : Dask seems to be faster, but you need to remember: `mean_da` isn't a number, it's a Dask array. Dask only **wrote the recipe**, which takes no time at all. This is the lazy trap: **a Dask cell that finishes instantly hasn't done any work yet.** To compare fairly, you have to count the compute time, too.
    
</div>

Below, you compare the file size of a numpy array and the Dask array.

In [ ]:
print("Shape of the numpy array : ", ones_np.shape) 
print("Shape of the dask array  : ", ones_da.shape) 

# memory size of numpy array in MiB
print(f"Memory size of numpy array in MB : {(ones_np.nbytes/ (1024**2)):.2f} MiB")
# memory size of dask array in MiB
print(f"Memory size of dask array in MB  : {var_size(ones_da):.2f} MiB")

<div class="alert alert-block alert-success">

**Reflection**

- ➡️ Why is the size of the Dask Array zero?
</div>

The Dask Array function `nbytes()` allows you to retrieve the size of the Dask Array.

In [ ]:
print("Size of Dask dataset:  {:.2f} MiB".format(ones_da.nbytes / 1024/1024))

### ➡️ Compute

Now compute and watch the **Task Stream** while it runs:

In [ ]:
%%time
mean_da.compute()

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. What did you see in the Task Stream?<br>
    ✏️ : Short coloured bars appear on several rows at the same time: your workers each creating a chunk of ones and averaging it, **in parallel**.
- ➡️ 2. Compare the time with `numpy`. Is Dask faster now?<br>
    ✏️ : Probably not by much, and perhaps not at all. 1 GB fits comfortably in memory, and `numpy` is very fast at this. Dask has to plan the tasks, hand them out and collect the results, and that organising costs time. **Dask isn't automatically faster.** It pays off when the data is bigger than memory, or when reading the data is slow, for example from the cloud.
</div>

### ➡️ Bigger than one worker's memory

This is where Dask really helps. Make the array **ten times longer**: about **9 GB**, more than a single worker's memory (7.8 GB).

<div class="alert alert-block alert-danger">

**⚠️ Don't try this with numpy**

`np.ones((100_000, 12_000))` would try to put all 9 GB into your notebook's memory at once. Only the Dask version is safe to run.
</div>

In [ ]:
big_da = da.ones((100_000, 12_000))
big_da

In [ ]:
%%time
big_da.mean().compute()

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. The array is bigger than any single worker's memory. Why didn't the **Workers Memory** bars fill up?<br>
    ✏️ : Each worker takes **one chunk at a time**: create a chunk of 128 MiB, average it, pass on a single small number, then clear the worktop for the next one. The full 9 GB never exists in one place. That is how Dask handles data larger than memory: **chunk by chunk, not all at once**.
- ➡️ 2. Why would `big_da.compute()`, without `.mean()`, be a bad idea?<br>
    ✏️ : `.compute()` brings the **whole result** back into your notebook. For `big_da`, that means all 9 GB in one place, which is exactly what Dask was helping you avoid. Always **reduce first** (a mean, a selection, a small map) and **compute last**.
</div>

The `numpy` array still takes up almost 1 GB of your notebook's memory. Free it before moving on:

In [ ]:
del ones_np

<div class="alert alert-block alert-info">

**Note**

📏 How big should a chunk be?

**About 128 MiB is a good default.** That is what Dask chooses when you leave the choice to it: big enough to be worth a task, small enough to fit easily on a worker. In the next section, the data comes from the cloud, already cut into chunks by someone else.
</div>

<hr>

<a id='example5'></a>

## Example 4️⃣ : Scale up with ERA5

You now return to ERA5, the store you opened in *Discover Zarr*, and calculate the **2m air temperature**: first for one year, then for ten year.

So far, you made the arrays yourself, so the chunks were yours to choose. Now the data comes from the cloud, already cut into chunks by the provider.

Keep the dashboard's **Task Stream**, **Progress** and **Workers Memory** panels in view.

### ➡️ 1. Chunking strategy of the data provider

Open ERA5 with `chunks={}`. This tells `xarray`: "use the chunks exactly as they are stored". You know these numbers from *Discover Zarr*:

In [ ]:
era5_url = "gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"

era5_stored = xr.open_zarr(era5_url, storage_options={"token": "anon"}, chunks={})
t2m_stored = era5_stored["2m_temperature"]

t2m_stored

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ Remember Exercise 1️⃣: more chunks means more tasks. How many tasks would Dask need just to *read* this variable? Is 0.9 MB a good portion size for your cooks?<br>
    ✏️ : Over **11,000 reading tasks**.. These portions are far too small. The storage chunks were chosen for whoever *stores* the data (here, fast access to any 2-day window of the whole globe), not for your calculation.

</div>

### ➡️ 2. Choose your working chunks

You can't change how the data is **stored**, but you can choose how Dask **groups** the stored chunks when it reads them. These groups are your **working chunks**. Ask for 1,024 time steps per chunk:

In [ ]:
era5 = xr.open_zarr(era5_url, storage_options={"token": "anon"}, chunks={"time": 1024})
t2m = era5["2m_temperature"]

t2m



Why 1,024? Two reasons:

* It is a **multiple of 8**, the stored time chunk. Each working chunk collects exactly 128 stored chunks. If you asked for 10 instead, some stored chunks would be split between two tasks and downloaded twice.
* It gives about **119 MB** per chunk, close to the 128 MiB that Dask chose for itself in Example 2️⃣.

<div class="alert alert-block alert-info">
**Note**

📦 Storage chunks and working chunks

- **Working chunk bigger than the storage chunk:** Dask groups several stored chunks into one. ✅ Fewer, bigger tasks.
- **Working chunk smaller than the storage chunk:** Dask must still download the *whole* stored chunk, and then uses only part of it. ❌ Wasted downloads.
- **Rule of thumb:** make your working chunks a **multiple** of the storage chunks, of around 100 MB.

In *Discover Zarr*, `.chunk()` decided how a store is divided on disk. Here, `chunks=` only decides how Dask groups the data while it works. Nothing in the store changes.
</div>

### ➡️ 3. Start small: one year

Select 2022 and write the recipe for the average map. It will be lazy, as always:

In [ ]:
t2m_1year = t2m.sel(time="2022")
mean_1year = t2m_1year.mean("time")

print(f"Data to read: {t2m_1year.nbytes / 1e6:.0f} MB in {t2m_1year.data.npartitions} chunks")

In [ ]:
%%time
mean_1year = mean_1year.compute()

### ➡️ 4. Scale up: ten years

Now ten times more data. The recipe is the same, only the selection changes:

In [ ]:
t2m_10years = t2m.sel(time=slice("2013", "2022"))
mean_10years = t2m_10years.mean("time")

print(f"Data to read: {t2m_10years.nbytes / 1e9:.2f} GB in {t2m_10years.data.npartitions} chunks")

In [ ]:
%%time
mean_10years = mean_10years.compute()

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. Ten times more data: did it take ten times longer?<br>
    ✏️ : No, much less than ten times. For one year, the work was shared between **only 2–3 chunks**, so only 2–3 threads were busy while the rest of the workers. Ten years give about 15 chunks, enough to keep all 8 threads busy.
- ➡️ 2. For a single year, how many rows of the Task Stream were busy? And for ten years?<br>
    ✏️ : That's the other side of chunk size: **big chunks are efficient, but you need enough of them to keep every worker busy.** A good number is at least a few chunks per thread.

</div>

### ➡️ 5. Prepare once, cook many times

So far, the workers have cleared their worktops after every compute. If you want several results from the same ten years, every `.compute()` downloads the data again, and downloading is the slow part. `.persist()` runs the recipe too, but **keeps the result on the workers**, ready to reuse:


| Questions | `.compute()` | `.persist()` |
|---|---|---|
| Runs the recipe | ✅ | ✅ |
| Where does the result go? | Into your notebook, as a `numpy` array or number | Stays on the workers, spread across their memory |
| What do you get back? | A normal value | A Dask array that is ready to use |
| Use it for… | Small final results: a number, a map, a time series | Data you will use several times, that fits in the cluster's memory |


About 1.7 GB fits easily on your workers' memory (about 31 GB in total). Persist the ten years. The cell returns at once, and the workers download in the background. Watch the **Workers Memory** bars rise, and wait until the Task Stream goes quiet:

In [ ]:
%%time
t2m_10years = t2m_10years.persist()

Now ask for three different results. Nothing has to be downloaded again:

In [ ]:
%%time
warmest = t2m_10years.max("time").compute()
coldest = t2m_10years.min("time").compute()
spread  = t2m_10years.std("time").compute()

### ➡️ 6. Look at the recipe, then the result

A real task graph can have hundreds of tasks. Use the **summary** view from Example 1️⃣. The full `.visualize()` would try to draw every box and can freeze your notebook:

In [ ]:
t2m.sel(time=slice("2013", "2022")).mean("time").data.dask.visualize()

And finally, the map. ERA5 stores temperature in kelvin, so subtract 273.15 for °C:

In [ ]:
(mean_10years - 273.15).plot(
    x="longitude", y="latitude",
    cmap="RdBu_r", center=0, size=5, aspect=2,
    cbar_kwargs={"label": "Mean 2 m temperature 2013–2022 (°C)"},
)

Free the memory before moving on:

In [ ]:
del t2m_10years

### 🎛️ The settings you control

Every Dask calculation in this notebook came down to the same few choices:

| Setting | Where you set it | What it changes |
|---|---|---|
| **Working chunks** | `chunks=` in `open_zarr()`, or `.chunk()` | Size and number of tasks: too small means lots of organising, too big means full worktops and idle cooks |
| **Selection** | `.sel(time=..., latitude=..., longitude=...)` | How much data is read |
| **Operation** | `.mean()`, `.max()`, `.groupby()`… | What each task does and how results are combined |
| **Compute or persist** | `.compute()` / `.persist()` | Whether results come back to you or stay on the workers |
| **Workers** | The **SCALE** button in the Dask sidebar | How many workers are available to work in parallel |


In the next section, you set three of these yourself.


<hr>

<a id='example6'></a>

## Exercise 2️⃣ : 64 Novembers at Snowbird

In *Discover Zarr*, you counted what the temperature at Snowbird would cost for all 64 years: 11,693 chunks, too many to read by hand. Now you have multiple workers. So ask a real question: **has November at Snowbird become warmer since 1959?** You need one average temperature for every November, from 1959 to 2022.

The calculation is written for you. **You** choose the settings: which years, and which working chunks. You run it three times and compare.

<div class="alert alert-block alert-warning">

**📝 Instructions**

- ➡️ 1. **Predict.** Runs B and C answer the same question, 64 Novembers. Which will be faster: B, with 92 big working chunks, or C, with 11,693 small ones?
- ➡️ 2. **Run A.** Run the parameters cell as it is, then the calculation cell. Note the chunks after selecting and the time.
- ➡️ 3. **Run B.** Change `years` to all 64 years and run both cells again. This one takes about a minute and a half, so watch the **Workers Memory** panel while you wait.
- ➡️ 4. **Run C.** Change `my_chunks` to `{}` and run both cells again.
- ➡️ 5. Run the plot cell.

**Hints**

- `{}` means "use the chunks exactly as they are stored".
- After every change, run the parameters cell **and** the calculation cell.
- Leave the calculation cell as it is.
</div>

### ➡️ Run A

In [ ]:
# Parameters: change these, then run this cell and the next one again
snowbird = dict(latitude=40.6, longitude=248.3, method="nearest")   # the same point as in Discover Zarr

years = slice("2013", "2022")      # runs B and C: slice("1959", "2022")
my_chunks = {"time": 1024}         # run C: {}

In [ ]:
%%time
era5 = xr.open_zarr(era5_url, storage_options={"token": "anon"}, chunks=my_chunks)
t2m = era5["2m_temperature"]

t2m_snowbird = t2m.sel(**snowbird).sel(time=years)                          # one grid point, your years
t2m_november = t2m_snowbird.sel(time=t2m_snowbird.time.dt.month == 11)      # only the Novembers
november_mean = t2m_november.groupby("time.year").mean() - 273.15           # one mean per year, in °C

print(f"Chunks after selecting: {t2m_november.data.npartitions}")
november_mean = november_mean.compute()

### ➡️ Run B

In [ ]:
# Parameters: change these, then run this cell and the next one again
snowbird = dict(latitude=40.6, longitude=248.3, method="nearest")   # the same point as in Discover Zarr

years = slice("1959", "2022")    
my_chunks = {"time": 1024}

In [ ]:
%%time
era5 = xr.open_zarr(era5_url, storage_options={"token": "anon"}, chunks=my_chunks)
t2m = era5["2m_temperature"]

t2m_snowbird = t2m.sel(**snowbird).sel(time=years)
t2m_november = t2m_snowbird.sel(time=t2m_snowbird.time.dt.month == 11)
november_mean = t2m_november.groupby("time.year").mean() - 273.15

print(f"Chunks to read: {t2m_november.data.npartitions}")
november_mean = november_mean.compute()

### ➡️ Run C

In [ ]:
# Parameters: change these, then run this cell and the next one again
snowbird = dict(latitude=40.6, longitude=248.3, method="nearest")   # the same point as in Discover Zarr

years = slice("1959", "2022")
my_chunks = {} 

In [ ]:
%%time
era5 = xr.open_zarr(era5_url, storage_options={"token": "anon"}, chunks=my_chunks)
t2m = era5["2m_temperature"]

t2m_snowbird = t2m.sel(**snowbird).sel(time=years)
t2m_november = t2m_snowbird.sel(time=t2m_snowbird.time.dt.month == 11)
november_mean = t2m_november.groupby("time.year").mean() - 273.15

print(f"Chunks to read: {t2m_november.data.npartitions}")
november_mean = november_mean.compute()

In [ ]:
november_mean.plot(marker="o", figsize=(10, 3))
plt.title("Mean November temperature near Snowbird")
plt.xlabel("")
plt.ylabel("Temperature (°C)")
plt.show()

<div class="alert alert-block alert-success">

**Reflection and answers**

- ➡️ 1. Compare runs B and C. Which was faster, and was your prediction right? Why?<br>
    ✏️ : **C was the fastest and B the slowest**, even though C has 960 small chunks and B only 8. The "chunks after selecting" hide the real cost. With `{"time": 1024}`, every reading task downloads a whole block of 1,024 time steps (about 8.5 months of the whole globe) and then **throws away** everything that isn't November. In run B, that means downloading almost the entire 11 GB variable. With the storage chunks, Dask only downloads the 15 small chunks per year that contain November: 960 chunks, less than 1 GB.
- ➡️ 2. From A to B, you asked for 6.4 times more years. What happened to the time?<br>
    ✏️ : The time grew by about **6.4 times** as well. More data simply means proportionally more time. Dask can share the work, but it can't make downloading free.

So, there's no single best chunk size. In Example 3️⃣, big chunks were ideal because you needed **every** time step for an average map. Here, you need only one month in twelve, so small chunks that match the selection win.
</div>

<hr>

<a id='example7'></a>

## Takeaways

You started with 16 numbers and a cluster with 4 works, and finished by asking 64 years of global weather how November at Snowbird has changed, without ever loading the whole archive into memory.

**What you learned**

* ⚙️ **A cluster has limits you can read.** The **scheduler** hands out tasks to the **workers**. Workers × threads is how many tasks run at once: 4 × 2 = 8 in this workshop. Memory per worker is the size of each worktop, and every chunk must fit on it with room to spare.
* 🧊 **Chunks become tasks, and Dask is lazy.** Every chunk creates tasks, and tasks that don't depend on each other run at the same time. Operations only write the **task graph** (the recipe), which is why they return instantly.
* ▶️ **`.compute()` or `.persist()`.** `.compute()` brings a result back into your notebook as a normal value. `.persist()` runs the recipe but keeps the data on the workers, ready to reuse without downloading it again.
* 📏 **Storage chunks and working chunks.** How data is *stored* is decided by whoever wrote it. How it is *processed* is up to you, with `chunks=` when you open it or `.chunk()` afterwards.
* 📈 **Scaling follows the data you read.** Once every worker is busy, more data means proportionally more time. The fastest way to scale is often to **read less**, not to add more workers.

**Things to keep in mind**

* **A Dask cell that finishes instantly hasn't done any work yet.** To compare speeds fairly, time the `.compute()`.
* **Dask isn't automatically faster.** For data that fits comfortably in memory, `numpy` is often quicker. Dask pays off when data is bigger than memory, or when reading it, from the cloud for example, is the slow part.
* **Reduce first, compute last.** Select, average or group while you are still lazy, and call `.compute()` only on a result small enough for your notebook.
* **Persist only what fits, and free it afterwards.** Keep an eye on the Workers Memory panel, and use `del` when you are done.
* **For big recipes, draw the summary.** Use `.dask.visualize()`, not `.visualize()`, which tries to draw every task and can freeze your notebook.

📚 *Further reading:* [Dask documentation](https://docs.dask.org) · [Dask array best practices](https://docs.dask.org/en/stable/array-best-practices.html) · [Xarray: parallel computing with Dask](https://docs.xarray.dev/en/stable/user-guide/dask.html) · [Understanding the Dask dashboard](https://docs.dask.org/en/stable/dashboard.html)

<hr>

&copy; 2026 thriveGEO GmbH
